# Kenya Climate Data Analysis (2015–2026)

**Objective:** Analyze Kenya's climate trends and compare with regional neighbors (Ethiopia, Sudan, Tanzania, Nigeria) for COP32 negotiations.

**Data Source:** NASA POWER climate reanalysis (January 2015 – March 2026)

This notebook follows the same structure as Ethiopia's EDA:
1. Data loading & parsing
2. Missing value handling
3. Outlier detection
4. Time series analysis
5. Correlation & relationships
6. Distribution analysis
7. Extreme events quantification
8. Policy-grade findings

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)
COUNTRY = 'Kenya'
CSV_PATH = os.path.join(DATA_DIR, 'kenya.csv')
CLEAN_PATH = os.path.join(DATA_DIR, 'kenya_clean.csv')
print(f"✓ Environment ready for {COUNTRY}")

In [ ]:
df = pd.read_csv(CSV_PATH)
df['Country'] = COUNTRY
df['DATE'] = pd.to_datetime(
    df['YEAR'].astype(str) + '-' + df['DOY'].astype(str).str.zfill(3),
    format='%Y-%j', errors='coerce'
)
df['Year'] = df['DATE'].dt.year
df['Month'] = df['DATE'].dt.month
print(f"Dataset: {df.shape}")
print(f"Date range: {df['DATE'].min()} to {df['DATE'].max()}")

In [ ]:
# Data cleaning
df = df.replace(-999.0, np.nan)
df = df.drop_duplicates().reset_index(drop=True)

# Missing value handling
threshold = int(df.shape[1] * 0.7)
df = df.dropna(thresh=threshold).reset_index(drop=True)

weather_vars = ['T2M', 'T2M_MAX', 'T2M_MIN', 'T2M_RANGE', 'PRECTOTCORR', 
                'RH2M', 'WS2M', 'WS2M_MAX', 'PS', 'QV2M']
df[weather_vars] = df[weather_vars].fillna(method='ffill').fillna(method='bfill')

print(f"✓ Cleaned dataset: {len(df)} rows")
print(f"✓ Remaining NaNs: {df.isna().sum().sum()}")

In [ ]:
# Summary statistics
print(df[['T2M', 'PRECTOTCORR', 'RH2M', 'WS2M']].describe().round(2))

In [ ]:
# Time series: Monthly temperature
df['YearMonth'] = df['DATE'].dt.to_period('M')
monthly = df.groupby('YearMonth').agg({
    'T2M': ['mean', 'std'],
    'PRECTOTCORR': 'sum'
}).reset_index()
monthly.columns = ['YearMonth', 'T2M_mean', 'T2M_std', 'PRECTOTCORR_sum']
monthly['YearMonth'] = monthly['YearMonth'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(monthly['YearMonth'], monthly['T2M_mean'], linewidth=2, marker='o', markersize=3, label='Monthly avg T2M')
ax.fill_between(monthly['YearMonth'], monthly['T2M_mean'] - monthly['T2M_std'], 
                 monthly['T2M_mean'] + monthly['T2M_std'], alpha=0.2)
ax.set_title(f'{COUNTRY} — Monthly Average Temperature (2015-2026)', fontsize=13, fontweight='bold')
ax.set_ylabel('Temperature (°C)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Temperature range: {monthly['T2M_mean'].min():.1f}°C to {monthly['T2M_mean'].max():.1f}°C")

In [ ]:
# Precipitation visualization
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(monthly['YearMonth'], monthly['PRECTOTCORR_sum'], width=20, alpha=0.7, edgecolor='black')
ax.set_title(f'{COUNTRY} — Monthly Precipitation (2015-2026)', fontsize=13, fontweight='bold')
ax.set_ylabel('Monthly precipitation (mm)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"Mean annual precipitation: {monthly.groupby(monthly['YearMonth'].dt.year)['PRECTOTCORR_sum'].sum().mean():.0f} mm")

In [ ]:
# Correlations
corr_vars = ['T2M', 'T2M_RANGE', 'PRECTOTCORR', 'RH2M', 'WS2M']
corr = df[corr_vars].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True, ax=ax)
ax.set_title(f'{COUNTRY} — Correlation Heatmap', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Extreme events
extreme_heat = (df['T2M_MAX'] > 35).sum()
heavy_rain = (df['PRECTOTCORR'] >= 50).sum()
dry_days = (df['PRECTOTCORR'] < 1).sum()

print(f"\n=== EXTREME EVENTS ===")
print(f"Days with T2M_MAX > 35°C: {extreme_heat} ({extreme_heat/len(df)*100:.1f}%)")
print(f"Heavy rain days (≥50 mm): {heavy_rain}")
print(f"Dry days (<1 mm): {dry_days} ({dry_days/len(df)*100:.1f}%)")

In [ ]:
# Export cleaned data
df.to_csv(CLEAN_PATH, index=False)
print(f"✓ Saved: {CLEAN_PATH}")

## Kenya Climate Findings

Add your policy-grade insights here (temperature trends, precipitation variability, extreme events, impacts on agriculture/water/energy, and COP32 policy asks).

Use the framework: What is changing? What did it cause? What does it demand?